# minispatial — Colab bootstrap

**Generated by `scripts/make_bootstrap_notebook.py`. Do not edit this notebook by hand** — edit the generator and re-run it.

Pinned commit: `2d49996ec5e6ca0daf9e2860ebe61788ceaffcc7`

Run the cells in order. Every step prints what it did; nothing important exists only in this notebook's output — results are written to JSON under `results/runs/` and copied to Drive.


## 1. Check the runtime

Record what hardware Colab gave you; it belongs in the run's provenance.

In [ ]:
import subprocess, sys, platform, json
print('python', platform.python_version())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU')


## 2. Mount Drive

Checkpoints and cached logits persist here; the Colab filesystem does not.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/minispatial'
import os; os.makedirs(DRIVE, exist_ok=True)
print('artifacts will be written to', DRIVE)


## 3. Clone the pinned commit

Pinning is what makes a result attributable to a repository state.

In [ ]:
COMMIT = '2d49996ec5e6ca0daf9e2860ebe61788ceaffcc7'
REPO   = 'https://github.com/AmeyaKI/minispatial.git'
!git clone -q $REPO /content/minispatial || true
%cd /content/minispatial
!git fetch -q origin && git checkout -q $COMMIT
!git rev-parse --short HEAD


## 4. Install

The `train` extra only. Core ML and MLX are Mac-side and are deliberately not installed here.

In [ ]:
!pip install -q -e '.[train]'
import importlib.metadata as md
for p in ('torch','terratorch','numpy'):
    print(p, md.version(p))


## 5. Sanity: the band contract resolves

If this fails, stop — every downstream number would be computed on the wrong bands.

In [ ]:
from minispatial.data.bands import load_band_spec
spec = load_band_spec()
print('bands:', spec.band_names)
print('constant_scale:', spec.constant_scale, 'ignore_index:', spec.ignore_index)
print('normalization from:', spec.normalization_source)


## 6. Get Sen1Floods11

Hand-labeled subset only: 446 chips, 1.02 GB. Resolved bucket and paths are in `DATA.md`. `/content` is ephemeral Colab scratch — it is wiped with the runtime, so nothing here touches the disk-location question for the Mac.

Re-running is safe: files already present at the right size are skipped, so a disconnect resumes rather than restarting.

In [ ]:
DATA_ROOT = '/content/sen1floods11'
!python scripts/download_sen1floods11.py --dest $DATA_ROOT \
    --json-out results/runs/sen1floods11_download.json


In [ ]:
from pathlib import Path
tifs = list(Path(DATA_ROOT).rglob('*.tif'))
csvs = list(Path(DATA_ROOT).rglob('*.csv'))
print(len(tifs), 'tif files,', len(csvs), 'split csvs')
assert len(tifs) == 892, f'expected 892 tifs (446 S2Hand + 446 LabelHand), got {len(tifs)}'
assert len(csvs) == 4, f'expected 4 split csvs, got {len(csvs)}'
print('dataset matches the survey in DATA.md')


## 7. Evaluate the 300M teacher — **the M0 gate**

This produces the number the whole project is gated on.

`--inference resize` mirrors the official recipe: the datamodule's own `albumentations.Resize(224,224)` is applied to image **and** mask, so metrics are computed at 224 against a downsampled mask. That is what the published recipe did, which is why it is the right mode for *reproducing* a published number.

**Before comparing:** read the published figure at its source and record *which mIoU definition the source states* alongside the number. Macro mIoU, IoU_water alone, and micro-averaged IoU differ by more than the proposed tolerance on a 2-class problem with this much class imbalance. A tolerance applied across two different definitions is not a gate.

Do not adjust the tolerance after seeing this number.

In [ ]:
!python train/eval.py --data-root $DATA_ROOT --split test \
    --inference resize --out results/runs/teacher_eval.json
import json; print(json.dumps(json.load(open('results/runs/teacher_eval.json')), indent=2))


In [ ]:
!mkdir -p $DRIVE/runs && cp results/runs/teacher_eval.json $DRIVE/runs/
print('copied teacher_eval.json to Drive')


## 8. Cache the teacher's test logits

fp16 plus a manifest with per-file SHA-256, so the M2 distillation run can prove which logits it consumed.

In [ ]:
!python train/cache_logits.py --data-root $DATA_ROOT --split test \
    --inference resize --out-dir $DRIVE/logits/test
!ls $DRIVE/logits/test | head -5


---

When these cells have run, bring `teacher_eval.json` back to the Mac and record the comparison in `RESULTS.md` — with the published figure's URL, not a remembered number.